# Groceries Product Recommendation Using Market Basket Analysis

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import re
import time
import math
import itertools
import numpy                      as np
import pandas                     as pd
import seaborn                    as sns
import plotly.express             as px
import matplotlib.pyplot          as plt

In [4]:
from   pyforest                   import *
from   mlxtend.preprocessing      import TransactionEncoder
from   mlxtend.frequent_patterns  import association_rules, fpgrowth

In [5]:
Df = pd.read_csv('C:\Khai phá dữ liệu/Retail_Transactions_Dataset2.csv')

<IPython.core.display.Javascript object>

In [6]:
df = Df.sample(n = 20000, random_state = 42)
df

,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion
96211,1000096211,44387.51111,Donna Cruz,"['Lawn Mower', 'Hair Gel', 'Apple', 'Apple', '...",7,69.50,Credit Card,Seattle,Pharmacy,True,Retiree,Fall,BOGO (Buy One Get One)
6469,1000006469,44143.45417,Courtney King,"['Cleaning Spray', 'Spinach', 'Canned Soup', '...",2,86.97,Debit Card,Houston,Supermarket,True,Retiree,Spring,BOGO (Buy One Get One)
26443,1000026443,44888.93264,Jennifer Ayala,"['Eggs', 'Laundry Detergent', 'Tea', 'Broom', ...",5,77.90,Debit Card,Atlanta,Convenience Store,False,Senior Citizen,Winter,NaN
76617,1000076617,44929.34375,Cynthia Thompson,"['Banana', 'Vinegar', 'Deodorant', 'Bath Towel...",1,23.17,Credit Card,Chicago,Specialty Store,True,Middle-Aged,Fall,Discount on Selected Items
71919,1000071919,44613.70625,Samantha Suarez,"['Hair Gel', 'Trash Cans', 'Carrots']",5,64.44,Mobile Payment,Houston,Pharmacy,True,Homemaker,Fall,Discount on Selected Items
...,...,...,...,...,...,...,...,...,...,...,...,...,...
168989,1000168989,44308.63472,Reginald Graham,"['Cleaning Rags', 'Tomatoes', 'Tuna', 'Baby Wi...",2,46.16,Cash,Atlanta,Pharmacy,True,Professional,Spring,NaN
75744,1000075744,44526.90972,Brittany Miller,"['Lawn Mower', 'Water']",7,32.36,Cash,Dallas,Supermarket,True,Middle-Aged,Spring,NaN
9948,1000009948,43871.84514,Allison Walker MD,"['Broom', 'Razors']",6,86.89,Cash,San Francisco,Warehouse Club,False,Young Adult,Winter,BOGO (Buy One Get One)
106436,1000106436,45083.90069,Trevor Sanchez,"['Banana', 'Soap', 'BBQ Sauce', 'Bath Towels']",1,73.57,Cash,New York,Pharmacy,True,Young Adult,Spring,BOGO (Buy One Get One)


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20000 entries, 96211 to 38161
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Transaction_ID     20000 non-null  int64  
 1   Date               20000 non-null  float64
 2   Customer_Name      20000 non-null  object 
 3   Product            20000 non-null  object 
 4   Total_Items        20000 non-null  int64  
 5   Total_Cost         20000 non-null  float64
 6   Payment_Method     20000 non-null  object 
 7   City               20000 non-null  object 
 8   Store_Type         20000 non-null  object 
 9   Discount_Applied   20000 non-null  bool   
 10  Customer_Category  20000 non-null  object 
 11  Season             20000 non-null  object 
 12  Promotion          13331 non-null  object 
dtypes: bool(1), float64(2), int64(2), object(8)
memory usage: 2.0+ MB


In [8]:
df2 = df[['Product']]
df2

,Product
96211,"['Lawn Mower', 'Hair Gel', 'Apple', 'Apple', '..."
6469,"['Cleaning Spray', 'Spinach', 'Canned Soup', '..."
26443,"['Eggs', 'Laundry Detergent', 'Tea', 'Broom', ..."
76617,"['Banana', 'Vinegar', 'Deodorant', 'Bath Towel..."
71919,"['Hair Gel', 'Trash Cans', 'Carrots']"
...,...
168989,"['Cleaning Rags', 'Tomatoes', 'Tuna', 'Baby Wi..."
75744,"['Lawn Mower', 'Water']"
9948,"['Broom', 'Razors']"
106436,"['Banana', 'Soap', 'BBQ Sauce', 'Bath Towels']"


In [9]:
df2['Product'] = df2['Product'].apply(lambda x: x.strip("[]").replace("'", ""))

In [10]:
df2.head(5)

,Product
96211,"Lawn Mower, Hair Gel, Apple, Apple, Ironing Board"
6469,"Cleaning Spray, Spinach, Canned Soup, Diapers"
26443,"Eggs, Laundry Detergent, Tea, Broom, Syrup"
76617,"Banana, Vinegar, Deodorant, Bath Towels, Mayon..."
71919,"Hair Gel, Trash Cans, Carrots"


In [37]:
df3 = df2['Product'].str.split(', ', expand=True)
df3

,0,1,2,3,4
96211,Lawn Mower,Hair Gel,Apple,Apple,Ironing Board
6469,Cleaning Spray,Spinach,Canned Soup,Diapers,None
26443,Eggs,Laundry Detergent,Tea,Broom,Syrup
76617,Banana,Vinegar,Deodorant,Bath Towels,Mayonnaise
71919,Hair Gel,Trash Cans,Carrots,None,None
...,...,...,...,...,...
168989,Cleaning Rags,Tomatoes,Tuna,Baby Wipes,Tuna
75744,Lawn Mower,Water,None,None,None
9948,Broom,Razors,None,None,None
106436,Banana,Soap,BBQ Sauce,Bath Towels,None


In [12]:
df3

,0,1,2,3,4
96211,Lawn Mower,Hair Gel,Apple,Apple,Ironing Board
6469,Cleaning Spray,Spinach,Canned Soup,Diapers,None
26443,Eggs,Laundry Detergent,Tea,Broom,Syrup
76617,Banana,Vinegar,Deodorant,Bath Towels,Mayonnaise
71919,Hair Gel,Trash Cans,Carrots,None,None
...,...,...,...,...,...
168989,Cleaning Rags,Tomatoes,Tuna,Baby Wipes,Tuna
75744,Lawn Mower,Water,None,None,None
9948,Broom,Razors,None,None,None
106436,Banana,Soap,BBQ Sauce,Bath Towels,None


In [13]:
df3.tail()

,0,1,2,3,4
168989,Cleaning Rags,Tomatoes,Tuna,Baby Wipes,Tuna
75744,Lawn Mower,Water,None,None,None
9948,Broom,Razors,None,None,None
106436,Banana,Soap,BBQ Sauce,Bath Towels,None
38161,Mop,Salmon,None,None,None


In [14]:
df3.shape

(20000, 5)

In [15]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20000 entries, 96211 to 38161
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       20000 non-null  object
 1   1       16009 non-null  object
 2   2       11976 non-null  object
 3   3       8011 non-null   object
 4   4       4071 non-null   object
dtypes: object(5)
memory usage: 937.5+ KB


### Exploratory Data Analysiss (EDA)

#### Tính tổng số lần xuất hiện của từng sản phẩm trong toàn bộ tập dữ liệu.

In [16]:
items_total = df3.apply(pd.Series.value_counts).sum(axis=1)
items_total = pd.DataFrame({'items': items_total.index, 'transactions': items_total.values})

items_total.sort_values('transactions',
                        ascending=False).head(10).reset_index(drop=True).style.background_gradient(cmap='Greens')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,items,transactions
0,Toothpaste,1522
1,Tuna,785
2,Cleaning Rags,780
3,Salmon,774
4,Trash Cans,767
5,Cereal Bars,766
6,Peanut Butter,766
7,Beef,766
8,Toothbrush,764
9,Broom,762


In [39]:
# kiểm tra sản phẩm trùng lặp

df3['jml_duplicate']       = df3.iloc[:, 0:5].apply(pd.Series.nunique, axis=1) 
df3['jml_product']    = df3.iloc[:, 0:5].count(axis=1) 
print('Số lượng giao dịch có sản phẩm trùng lặp:', len(df3[df3['jml_duplicate'] != df3['jml_product']]))

<IPython.core.display.Javascript object>

Số lượng giao dịch có sản phẩm trùng lặp: 955


In [43]:
print('Số lượng giao dịch có sản phẩm trùng lặp:', len(df3[df3['jml_duplicate'] != df3['jml_product']]))

Số lượng giao dịch có sản phẩm trùng lặp: 955


In [40]:
df3

,0,1,2,3,4,jml_duplicate,jml_product
96211,Lawn Mower,Hair Gel,Apple,Apple,Ironing Board,4,5
6469,Cleaning Spray,Spinach,Canned Soup,Diapers,None,4,4
26443,Eggs,Laundry Detergent,Tea,Broom,Syrup,5,5
76617,Banana,Vinegar,Deodorant,Bath Towels,Mayonnaise,5,5
71919,Hair Gel,Trash Cans,Carrots,None,None,3,3
...,...,...,...,...,...,...,...
168989,Cleaning Rags,Tomatoes,Tuna,Baby Wipes,Tuna,4,5
75744,Lawn Mower,Water,None,None,None,2,2
9948,Broom,Razors,None,None,None,2,2
106436,Banana,Soap,BBQ Sauce,Bath Towels,None,4,4


In [48]:
df_clean = df3[df3['jml_duplicate'] == df3['jml_product']].iloc[:, 0:5]
print('Số lượng giao dịch không có sản phẩm trùng lặp:', len(df_clean))

Số lượng giao dịch không có sản phẩm trùng lặp: 19045


In [22]:
df_clean

,0,1,2,3,4
6469,Cleaning Spray,Spinach,Canned Soup,Diapers,None
26443,Eggs,Laundry Detergent,Tea,Broom,Syrup
76617,Banana,Vinegar,Deodorant,Bath Towels,Mayonnaise
71919,Hair Gel,Trash Cans,Carrots,None,None
29279,Syrup,Peanut Butter,Broom,Lawn Mower,None
...,...,...,...,...,...
168989,Cleaning Rags,Tomatoes,Tuna,Baby Wipes,Tuna
75744,Lawn Mower,Water,None,None,None
9948,Broom,Razors,None,None,None
106436,Banana,Soap,BBQ Sauce,Bath Towels,None


In [23]:
df_clean = df_clean.fillna(value= float('nan'))
df_clean

,0,1,2,3,4
6469,Cleaning Spray,Spinach,Canned Soup,Diapers,NaN
26443,Eggs,Laundry Detergent,Tea,Broom,Syrup
76617,Banana,Vinegar,Deodorant,Bath Towels,Mayonnaise
71919,Hair Gel,Trash Cans,Carrots,NaN,NaN
29279,Syrup,Peanut Butter,Broom,Lawn Mower,NaN
...,...,...,...,...,...
168989,Cleaning Rags,Tomatoes,Tuna,Baby Wipes,Tuna
75744,Lawn Mower,Water,NaN,NaN,NaN
9948,Broom,Razors,NaN,NaN,NaN
106436,Banana,Soap,BBQ Sauce,Bath Towels,NaN


In [24]:
items_total_clean = df_clean.apply(pd.Series.value_counts).sum(axis=1)
items_total_clean = pd.DataFrame({'item': items_total_clean.index, 'transaction': items_total_clean.values})

items_total_clean.sort_values('transaction',
                               ascending=False).head(10).reset_index(drop = True).style.background_gradient(cmap='Greens')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,item,transaction
0,Toothpaste,1395
1,Tuna,742
2,Cleaning Rags,740
3,Peanut Butter,733
4,Trash Cans,732
5,Dishware,730
6,Cereal Bars,729
7,Soap,726
8,Toothbrush,725
9,Mustard,725


### Algoritma FP-Growth

In [25]:
#xác định nhưỡng tối thiểu
min_supp = 0.002
min_conf = 0.01

In [26]:
df7 = df_clean.replace(np.nan, None)
df7.head(5)

<IPython.core.display.Javascript object>

,0,1,2,3,4
6469,Cleaning Spray,Spinach,Canned Soup,Diapers,None
26443,Eggs,Laundry Detergent,Tea,Broom,Syrup
76617,Banana,Vinegar,Deodorant,Bath Towels,Mayonnaise
71919,Hair Gel,Trash Cans,Carrots,None,None
29279,Syrup,Peanut Butter,Broom,Lawn Mower,None


In [45]:
transaction2 = []

for _, row in df7.iterrows():
    temp = [item for item in row if item is not None]
    transaction2.append(temp)
    
transaction2[0:5]

[['Cleaning Spray', 'Spinach', 'Canned Soup', 'Diapers'],
 ['Eggs', 'Laundry Detergent', 'Tea', 'Broom', 'Syrup'],
 ['Banana', 'Vinegar', 'Deodorant', 'Bath Towels', 'Mayonnaise'],
 ['Hair Gel', 'Trash Cans', 'Carrots'],
 ['Syrup', 'Peanut Butter', 'Broom', 'Lawn Mower']]

In [46]:
encode_tran2  = TransactionEncoder()
array_tran_fp = encode_tran2.fit(transaction2).transform(transaction2)

df8 = pd.DataFrame(array_tran_fp, columns=encode_tran2.columns_)
df8.head(5)

<IPython.core.display.Javascript object>

,Air Freshener,Apple,BBQ Sauce,Baby Wipes,Banana,Bath Towels,Beef,Bread,Broom,Butter,...,Tomatoes,Toothbrush,Toothpaste,Trash Bags,Trash Cans,Tuna,Vacuum Cleaner,Vinegar,Water,Yogurt
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,True,True,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
4,False,False,False,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False


In [29]:
start_time = time.time()
fpg         = fpgrowth(df8,
                       min_support  = min_supp,
                       use_colnames = True)

elapsed_time_fpgrowth = time.time() - start_time

In [47]:
start_time = time.time()

rule_FP     = association_rules(fpg,
                                metric="confidence",
                                min_threshold=min_conf) #0.01

rule_FP     = rule_FP[rule_FP.lift > 1]
rule_FP     = rule_FP.reset_index(drop=True)

elapsed_time_fpgrowth += time.time() - start_time

rule_FP.head(5)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Toothpaste),(Canned Soup),0.072168,0.035200,0.002600,0.036023,1.023385,1.0,0.000059,1.000854,0.024628,0.024814,0.000853,0.054939
1,(Canned Soup),(Toothpaste),0.035200,0.072168,0.002600,0.073855,1.023385,1.0,0.000059,1.001822,0.023684,0.024814,0.001819,0.054939
2,(Eggs),(Toothpaste),0.036396,0.072168,0.002912,0.080000,1.108530,1.0,0.000285,1.008513,0.101603,0.027559,0.008442,0.060173
3,(Toothpaste),(Eggs),0.072168,0.036396,0.002912,0.040346,1.108530,1.0,0.000285,1.004116,0.105520,0.027559,0.004099,0.060173
4,(Toothpaste),(Deodorant),0.072168,0.035824,0.002860,0.039625,1.106117,1.0,0.000274,1.003958,0.103398,0.027201,0.003943,0.059726


### Recommendation Product

In [32]:
def product_recommendations(*args):
    filtered_rules = rule_FP[rule_FP['antecedents'].apply(lambda x: all(item in x for item in args))]
    
    sorted_rules = filtered_rules.sort_values(by='confidence', ascending=False).head(10)
    
    sorted_rules['antecedents'] = sorted_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
    sorted_rules['consequents'] = sorted_rules['consequents'].apply(lambda x: ', '.join(list(x)))
    
    fig = px.bar(sorted_rules,
                 x='consequents',
                 y='confidence',
                 color='confidence',
                 color_continuous_scale='Greens',
                 title=f'Product Recommendations Based on Confidence for Products: {", ".join(args)}')
    
    fig.update_layout(xaxis_title='product_recommendations (Consequents)',
                      yaxis_title='Confidence',
                      xaxis_tickangle=-45)
    
    fig.show()

In [33]:
recommendations2 = product_recommendations('Toothpaste')
recommendations2

<IPython.core.display.Javascript object>